# BioTrace - Demostración de Uso

Este notebook demuestra cómo utilizar el **BioTraceDAO** para interactuar con la base de datos, extraer información y analizarla utilizando **Pandas** y **Matplotlib**, tal como se solicita en el proyecto integrador.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dao import BioTraceDAO

# Inicializamos el DAO
dao = BioTraceDAO()
print("Conectado a la base de datos BioTrace")

## 1. Listar Pacientes

In [ ]:
pacientes = dao.listar_pacientes(limit=5)

print(f"Se encontraron {len(pacientes)} pacientes. Mostrando los primeros 5:\n")
for p in pacientes:
    print(f"- {p.nombre} {p.apellido} (DNI: {p.dni}) - {p.genero}")

## 2. Análisis Agrupados (Aggregation Pipeline)
Vamos a ver cuántos análisis se realizaron por cada tipo.

In [ ]:
reporte_tipos = dao.reporte_analisis_por_tipo()

# Convertimos a DataFrame de Pandas para mejor visualización
df_tipos = pd.DataFrame(reporte_tipos)
df_tipos.rename(columns={'_id': 'Tipo de Análisis', 'total': 'Cantidad'}, inplace=True)
df_tipos

In [ ]:
# Visualizamos con matplotlib
if not df_tipos.empty:
    plt.figure(figsize=(10, 5))
    plt.bar(df_tipos['Tipo de Análisis'], df_tipos['Cantidad'], color='skyblue')
    plt.title('Cantidad de Análisis por Tipo')
    plt.xlabel('Tipo de Análisis')
    plt.ylabel('Cantidad')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Trazabilidad Completa de un Paciente
Usaremos la consulta de agregación `$lookup` (JOIN en MongoDB) para traer las muestras y sus análisis asociados en un solo viaje.

In [ ]:
if pacientes:
    paciente_ejemplo_id = pacientes[0].id
    trazabilidad = dao.reporte_trazabilidad_paciente(paciente_ejemplo_id)
    
    print(f"Trazabilidad para paciente ID: {paciente_ejemplo_id}\n")
    
    for muestra in trazabilidad:
        print(f"Muestra: {muestra['tipo_muestra']} - Estado: {muestra['estado']}")
        analisis = muestra.get('analisis_realizados', [])
        if analisis:
            for a in analisis:
                print(f"  └─ Análisis: {a['tipo_analisis']} (Score: {a['metricas_calidad'].get('q_score', 'N/A')})")
        else:
            print("  └─ Sin análisis registrados aún.")
else:
    print("No hay pacientes registrados. Ejecuta 'seed.py' primero.")